In [3]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 2000

patients = pd.DataFrame({
    "patient_id": range(1, n + 1),

    "age": np.random.normal(
        loc=32,
        scale=5,
        size=n
    ).round(1),

    "bmi": np.random.normal(
        loc=31,
        scale=6,
        size=n
    ).round(1),

    "diabetes": np.random.binomial(
        1,
        0.45,
        size=n
    ),

    "hypertension": np.random.binomial(
        1,
        0.20,
        size=n
    ),

    "smoking": np.random.binomial(
        1,
        0.15,
        size=n
    ),

    "prior_preterm_birth": np.random.binomial(
        1,
        0.10,
        size=n
    )
})

patients.head()

,patient_id,age,bmi,diabetes,hypertension,smoking,prior_preterm_birth
0,1,34.5,26.9,0,0,0,0
1,2,31.3,30.1,0,0,0,0
2,3,35.2,26.2,1,0,0,0
3,4,39.6,29.2,0,0,0,0
4,5,30.8,19.6,1,0,0,0


In [4]:
# Probability of late GLP-1RA discontinuation

linear_score = (
    -0.3
    + 0.8 * patients["diabetes"]
    + 0.04 * (patients["bmi"] - 30)
    + 0.4 * patients["hypertension"]
    + 0.3 * patients["prior_preterm_birth"]
)

prob_late = 1 / (1 + np.exp(-linear_score))

patients["late_discontinuation"] = np.random.binomial(
    1,
    prob_late
)

patients["days_before_conception"] = np.where(
    patients["late_discontinuation"] == 1,
    np.random.randint(0, 61, size=n),
    np.random.randint(61, 181, size=n)
)

patients["treatment_strategy"] = np.where(
    patients["late_discontinuation"] == 1,
    "Late discontinuation",
    "Early discontinuation"
)

patients.head()

,patient_id,age,bmi,diabetes,hypertension,smoking,prior_preterm_birth,late_discontinuation,days_before_conception,treatment_strategy
0,1,34.5,26.9,0,0,0,0,0,169,Early discontinuation
1,2,31.3,30.1,0,0,0,0,0,177,Early discontinuation
2,3,35.2,26.2,1,0,0,0,0,141,Early discontinuation
3,4,39.6,29.2,0,0,0,0,1,45,Late discontinuation
4,5,30.8,19.6,1,0,0,0,0,85,Early discontinuation


In [5]:
patients["treatment_strategy"].value_counts()

,count
treatment_strategy,
Late discontinuation,1117
Early discontinuation,883


In [6]:
patients["treatment_strategy"].value_counts(normalize=True).round(3)

,proportion
treatment_strategy,
Late discontinuation,0.558
Early discontinuation,0.442


In [7]:
# Probability of preterm birth

outcome_score = (
    -2.4
    + 0.45 * patients["late_discontinuation"]
    + 0.55 * patients["diabetes"]
    + 0.03 * (patients["bmi"] - 30)
    + 0.40 * patients["hypertension"]
    + 0.25 * patients["smoking"]
    + 0.70 * patients["prior_preterm_birth"]
    + 0.02 * (patients["age"] - 32)
)

prob_preterm = 1 / (1 + np.exp(-outcome_score))

patients["preterm_birth"] = np.random.binomial(
    1,
    prob_preterm
)

patients.head()

,patient_id,age,bmi,diabetes,hypertension,smoking,prior_preterm_birth,late_discontinuation,days_before_conception,treatment_strategy,preterm_birth
0,1,34.5,26.9,0,0,0,0,0,169,Early discontinuation,0
1,2,31.3,30.1,0,0,0,0,0,177,Early discontinuation,0
2,3,35.2,26.2,1,0,0,0,0,141,Early discontinuation,1
3,4,39.6,29.2,0,0,0,0,1,45,Late discontinuation,0
4,5,30.8,19.6,1,0,0,0,0,85,Early discontinuation,0


In [8]:
patients.groupby("treatment_strategy")["preterm_birth"].agg(
    ["count", "sum", "mean"]
).round(3)

,count,sum,mean
treatment_strategy,,,
Early discontinuation,883,112,0.127
Late discontinuation,1117,237,0.212


In [9]:
risk_early = patients.loc[
    patients["late_discontinuation"] == 0,
    "preterm_birth"
].mean()

risk_late = patients.loc[
    patients["late_discontinuation"] == 1,
    "preterm_birth"
].mean()

risk_difference = risk_late - risk_early
risk_ratio = risk_late / risk_early

print("Early risk:", round(risk_early, 3))
print("Late risk:", round(risk_late, 3))
print("Crude risk difference:", round(risk_difference, 3))
print("Crude risk ratio:", round(risk_ratio, 2))

Early risk: 0.127
Late risk: 0.212
Crude risk difference: 0.085
Crude risk ratio: 1.67


In [10]:
baseline_vars = [
    "age",
    "bmi",
    "diabetes",
    "hypertension",
    "smoking",
    "prior_preterm_birth"
]

baseline_table = patients.groupby(
    "treatment_strategy"
)[baseline_vars].mean().round(3)

baseline_table

,age,bmi,diabetes,hypertension,smoking,prior_preterm_birth
treatment_strategy,,,,,,
Early discontinuation,32.087,30.432,0.324,0.156,0.156,0.082
Late discontinuation,32.336,31.378,0.519,0.204,0.147,0.126


In [11]:
from sklearn.linear_model import LogisticRegression

X = patients[baseline_vars]
y = patients["late_discontinuation"]

ps_model = LogisticRegression(max_iter=1000)

ps_model.fit(X, y)

patients["propensity_score"] = ps_model.predict_proba(X)[:, 1]

patients[
    ["patient_id", "late_discontinuation", "propensity_score"]
].head()

,patient_id,late_discontinuation,propensity_score
0,1,0,0.433510
1,2,0,0.440853
2,3,0,0.632030
3,4,1,0.469409
4,5,0,0.573387


In [12]:
patients["iptw"] = np.where(
    patients["late_discontinuation"] == 1,
    1 / patients["propensity_score"],
    1 / (1 - patients["propensity_score"])
)

patients[
    [
        "patient_id",
        "late_discontinuation",
        "propensity_score",
        "iptw"
    ]
].head()

,patient_id,late_discontinuation,propensity_score,iptw
0,1,0,0.433510,1.765255
1,2,0,0.440853,1.788438
2,3,0,0.632030,2.717612
3,4,1,0.469409,2.130337
4,5,0,0.573387,2.344043


In [13]:
patients["iptw"].describe().round(3)

,iptw
count,2000.000
mean,2.001
std,0.577
min,1.179
25%,1.599
50%,1.860
75%,2.243
max,5.889


In [14]:
early = patients[patients["late_discontinuation"] == 0]
late = patients[patients["late_discontinuation"] == 1]

balance_comparison = pd.DataFrame({

    "Early_before": early[baseline_vars].mean(),

    "Late_before": late[baseline_vars].mean(),

    "Early_after_IPTW": [
        np.average(
            early[var],
            weights=early["iptw"]
        )
        for var in baseline_vars
    ],

    "Late_after_IPTW": [
        np.average(
            late[var],
            weights=late["iptw"]
        )
        for var in baseline_vars
    ]
})

balance_comparison.round(3)

,Early_before,Late_before,Early_after_IPTW,Late_after_IPTW
age,32.087,32.336,32.236,32.239
bmi,30.432,31.378,31.034,30.983
diabetes,0.324,0.519,0.435,0.435
hypertension,0.156,0.204,0.190,0.185
smoking,0.156,0.147,0.146,0.148
prior_preterm_birth,0.082,0.126,0.106,0.107


In [15]:
 def weighted_mean_var(x, w):
    mean = np.average(x, weights=w)
    var = np.average((x - mean) ** 2, weights=w)
    return mean, var


def smd_unweighted(var):
    early_var = early[var]
    late_var = late[var]

    mean_early = early_var.mean()
    mean_late = late_var.mean()

    var_early = early_var.var()
    var_late = late_var.var()

    pooled_sd = np.sqrt((var_early + var_late) / 2)

    return (mean_late - mean_early) / pooled_sd


def smd_weighted(var):
    mean_early, var_early = weighted_mean_var(
        early[var],
        early["iptw"]
    )

    mean_late, var_late = weighted_mean_var(
        late[var],
        late["iptw"]
    )

    pooled_sd = np.sqrt((var_early + var_late) / 2)

    return (mean_late - mean_early) / pooled_sd


smd_table = pd.DataFrame({
    "SMD_before": [
        smd_unweighted(var)
        for var in baseline_vars
    ],

    "SMD_after_IPTW": [
        smd_weighted(var)
        for var in baseline_vars
    ]
}, index=baseline_vars)

smd_table.round(3)

,SMD_before,SMD_after_IPTW
age,0.050,0.001
bmi,0.157,-0.008
diabetes,0.403,-0.000
hypertension,0.125,-0.011
smoking,-0.026,0.008
prior_preterm_birth,0.147,0.003


In [16]:
weighted_risk_early = np.average(
    early["preterm_birth"],
    weights=early["iptw"]
)

weighted_risk_late = np.average(
    late["preterm_birth"],
    weights=late["iptw"]
)

weighted_risk_difference = (
    weighted_risk_late - weighted_risk_early
)

weighted_risk_ratio = (
    weighted_risk_late / weighted_risk_early
)

print(
    "Weighted early risk:",
    round(weighted_risk_early, 3)
)

print(
    "Weighted late risk:",
    round(weighted_risk_late, 3)
)

print(
    "IPTW risk difference:",
    round(weighted_risk_difference, 3)
)

print(
    "IPTW risk ratio:",
    round(weighted_risk_ratio, 2)
)

Weighted early risk: 0.143
Weighted late risk: 0.199
IPTW risk difference: 0.056
IPTW risk ratio: 1.39


In [17]:
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(42)

bootstrap_rd = []
bootstrap_rr = []

n_boot = 500

for i in range(n_boot):

    # Resample patients with replacement
    idx = rng.choice(
        len(patients),
        size=len(patients),
        replace=True
    )

    boot = patients.iloc[idx].copy()

    # Re-estimate propensity scores
    X_boot = boot[baseline_vars]
    y_boot = boot["late_discontinuation"]

    model = LogisticRegression(max_iter=1000)
    model.fit(X_boot, y_boot)

    ps = model.predict_proba(X_boot)[:, 1]

    # Avoid extremely small or large probabilities
    ps = np.clip(ps, 0.01, 0.99)

    boot["iptw_boot"] = np.where(
        boot["late_discontinuation"] == 1,
        1 / ps,
        1 / (1 - ps)
    )

    early_boot = boot[
        boot["late_discontinuation"] == 0
    ]

    late_boot = boot[
        boot["late_discontinuation"] == 1
    ]

    risk_early_boot = np.average(
        early_boot["preterm_birth"],
        weights=early_boot["iptw_boot"]
    )

    risk_late_boot = np.average(
        late_boot["preterm_birth"],
        weights=late_boot["iptw_boot"]
    )

    bootstrap_rd.append(
        risk_late_boot - risk_early_boot
    )

    bootstrap_rr.append(
        risk_late_boot / risk_early_boot
    )


rd_ci = np.percentile(
    bootstrap_rd,
    [2.5, 97.5]
)

rr_ci = np.percentile(
    bootstrap_rr,
    [2.5, 97.5]
)

print(
    "IPTW Risk Difference:",
    round(weighted_risk_difference, 3),
    "95% CI:",
    np.round(rd_ci, 3)
)

print(
    "IPTW Risk Ratio:",
    round(weighted_risk_ratio, 2),
    "95% CI:",
    np.round(rr_ci, 2)
)

IPTW Risk Difference: 0.056 95% CI: [0.025 0.09 ]
IPTW Risk Ratio: 1.39 95% CI: [1.15 1.71]


In [18]:
results_table = pd.DataFrame({
    "Analysis": [
        "Crude",
        "IPTW-adjusted"
    ],

    "Early risk (%)": [
        risk_early * 100,
        weighted_risk_early * 100
    ],

    "Late risk (%)": [
        risk_late * 100,
        weighted_risk_late * 100
    ],

    "Risk difference (percentage points)": [
        risk_difference * 100,
        weighted_risk_difference * 100
    ],

    "RD 95% CI": [
        "Not calculated",
        f"{rd_ci[0] * 100:.1f} to {rd_ci[1] * 100:.1f}"
    ],

    "Risk ratio": [
        risk_ratio,
        weighted_risk_ratio
    ],

    "RR 95% CI": [
        "Not calculated",
        f"{rr_ci[0]:.2f} to {rr_ci[1]:.2f}"
    ]
})

results_table.round(2)

,Analysis,Early risk (%),Late risk (%),Risk difference (percentage points),RD 95% CI,Risk ratio,RR 95% CI
0,Crude,12.68,21.22,8.53,Not calculated,1.67,Not calculated
1,IPTW-adjusted,14.29,19.88,5.58,2.5 to 9.0,1.39,1.15 to 1.71


In [19]:
# Save synthetic dataset and HTA-style results

patients.to_csv(
    "synthetic_glp1_pregnancy.csv",
    index=False
)

results_table.to_csv(
    "hta_results_summary.csv",
    index=False
)

print("Files saved successfully.")

Files saved successfully.


In [20]:
from google.colab import files

files.download("synthetic_glp1_pregnancy.csv")
files.download("hta_results_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>